# BEAVER-ES: Time Series
***

***Author**: Chus Casado Rodríguez*<br>
***Date**: 17-09-2026*<br>

**Introduction:**<br>



**Outputs:**<br>


**To do**:<br>
* [] Filter stations with wrong catchment polygon and fix it.
* [x] Probably the timestamps in CERRA are shifted one day (like EMO1).
* [x] How to trim the meteo time series at the start? Should I include one extra year as initial condition for the first discharge observation?

In [1]:
from pathlib import Path
from tqdm.auto import tqdm
import json
import logging
logger = logging.getLogger(__name__)

import pandas as pd
import geopandas as gpd
from sklearn.model_selection import train_test_split

from ocab.config import Config
from ocab.timeseries.build_dataset import combine_periods, time_encoding, valid_timeseries
from ocab.utils.sampling import create_sample_file, create_period_file

## Configuration

In [2]:
cfg = Config('config_BEAVERS_v100.yml')

# point layer
txt_file = 'dams.geojson'
headwater_only = True

# paths
path_in = Path('../../docs/timeseries/reservoirs')

# questionnaire
url_form = 'https://docs.google.com/spreadsheets/d/e/2PACX-1vQ-uWba6lG8gvXwSd5kMmU2lHp8hgPG4agc8LhikPO-jQdbWHZCeMYFbLtbo6jW-436SgRenZoxiaaH/pub?gid=1682822662&single=true&output=csv'

# import timeseries metadata
with open('metadata.json', 'r', encoding='utf-8') as f:
    metadata = json.load(f)

# required variables
key_vars = ['filling']

## Data

### Dams

In [4]:
# load points
points = gpd.read_file(cfg.path_gis / txt_file).set_index('id')
print(f'no. points: {len(points):4}')

# identify basins
basins = points['basin'].unique()
print(f'no. basins: {len(basins):4}')

no. points:  366
no. basins:   12


### Selection

I load and handle the answers to the questionnaire in the [website](https://casadoj.github.io/of_camels_and_beavers/).

In [5]:
# load answers to the online questionnaire
answers = pd.read_csv(url_form, parse_dates=True)
print(f'No. answers:\t\t{len(answers)}\n')

# rename columns
rename_cols = {
    'Marca temporal': 'timestamp', 
    'Reservoir ID': 'ID', 
    'Gauging stations directly upstream the reservoir.': 'gauge_upstream',
    'Gauging stations directly downstream the reservoir.': 'gauge_downstream',
    'Reservoir directly downstream.': 'reservoir_downstream',
    'Is the catchment polygon correct?': 'catchment',
    'Is any of these time series clearly incorrect?': 'incorrect_ts',
    'Start date (1st period)': 'start_1', 
    'End date (1st period)': 'end_1',
    'Is there a second period of high-quality data?': 'second_period',
    'Start date (2nd period)': 'start_2', 
    'End date (2nd period)': 'end_2',
    'Is there a third period of high-quality data?': 'third_period',
    'Start date (3rd period)': 'start_3', 
    'End date (3rd period)': 'end_3',
    'Is the indicated reservoir use correct?': 'use_correct',
    "What's the correct main use of the reservoir?": 'main_use',
    'Raise any other issue in the reservoir attributes or time series. E.g., filling values above 1 may indicate an error in the storage capacity.': 'comments',
    # 'email', 
}
answers.rename(columns=rename_cols, inplace=True)

# Check if all points have been revised
for basin in basins:
    missing = points[points['basin'] == basin].index.difference(answers['ID'])
    if len(missing > 0):
        print('{0:<12}: {1}'.format(basin, list(missing)))

# keep only selected points
answers = answers[answers['ID'].isin(points.index)]
print('Raw data')
print(f'No. answers:\t\t{len(answers)}')
print(f'No. unique points:\t{len(answers["ID"].unique())}')

# if duplicate points, keep only the last answer
answers = answers.sort_values('timestamp').drop_duplicates('ID', keep='last')
answers = answers.set_index('ID', drop=True).sort_index(axis=0)

# keep reservoirs with time series for the key variables
valid_ts = valid_timeseries(answers)
answers = pd.concat([
    answers.drop('incorrect_ts', axis=1),
    valid_timeseries(answers, column='incorrect_ts')
], axis=1)

# answers = answers[~(valid_ts == 0).all(axis=1)]
mask_vars = answers[key_vars].all(axis=1)
answers = answers[mask_vars]
print(f'\nNo. points with key variables: {mask_vars.sum()}')

# combine selected periods
answers[['start_dates', 'end_dates']] = answers.apply(combine_periods, axis=1)

# drop some columns
drop = ['timestamp', 'start_1', 'end_1', 'second_period', 'start_2', 'end_2', 'third_period', 'start_3', 'end_3']
answers.drop(columns=drop, inplace=True)

No. answers:		384

Raw data
No. answers:		376
No. unique points:	366

No. points with key variables: 341


## Export

### Samples and Periods

In [6]:
# define output folder
path_samples = cfg.path_dataset / 'selection'
path_samples.mkdir(exist_ok=True)
print(f'Samples will be saved in {path_samples}')

# keep points selected in the questionnaire
points = points.loc[points.index.intersection(answers.index)]


Samples will be saved in /home/casadoj/Data/BEAVERS-ES/v1_0_0/selection


In [7]:
# list of headwater reservoirs
if headwater_only:
    selection = pd.read_csv(cfg.path_dataset / 'selection' / 'headwater_reservoirs.txt', header=None).squeeze().to_list()
    selection = [int(ID.split('_')[1]) for ID in selection]
    print(f'{len(selection)} headwater reservoirs')
    selection = points.index.intersection(selection)
else:
    selection = points.index.to_list()
print(f'{len(selection)} reservoirs in the selection')

218 headwater reservoirs
203 reservoirs in the selection


In [8]:

# only because SEGURA has few stations!!
points['stratify'] = points['basin']
# merge_group = points['basin'].isin(['JUCAR', 'SEGURA'])
# points.loc[merge_group, 'stratify'] = 'MEDITERRÁNEO'
merge_group = points['basin'].isin(['GALICIA COSTA', 'MIÑO-SIL'])
points.loc[merge_group, 'stratify'] = 'ATLÁNTICO'

# divide basins in train, validation and test sets
train_set, temp = train_test_split(
    points.loc[selection],
    train_size=cfg.train_size,
    random_state=cfg.seed,
    stratify=points.loc[selection, 'stratify']
)
val_set, test_set = train_test_split(
    temp,
    train_size=cfg.val_size / (1 - cfg.train_size),
    random_state=cfg.seed,
    stratify=temp['stratify']
)
print(f'train size:\t\t{len(train_set)}')
print(f'validation size:\t{len(val_set)}')
print(f'test size:\t\t{len(test_set)}')

# organize basin IDs according to the sample
samples = {
    'train': train_set.sort_index().index.to_list(),
    'validation': val_set.sort_index().index.to_list(),
    'test': test_set.sort_index().index.to_list(),
}

# create sample files
for name, sample in samples.items():
    if headwater_only:
        txt_file = path_samples / f'headwater_{name}.txt'
        pkl_file = path_samples / f'periods_headwaater_{name}.pkl'
    else:
        txt_file = path_samples / f'reservoirs_{name}.txt'
        pkl_file = path_samples / f'periods_{name}.pkl'
    # export TXT file

    # all dataset
    create_sample_file(cfg, sample, txt_file)

    # # per basin
    # for basin in basins:
    #     sample_basin = points[points['basin'] == basin].index.intersection(sample).tolist()
    #     if len(sample_basin) > 0:
    #         create_sample_file(cfg, sample_basin, path_samples / f'{basin.lower()}_{name}.txt')

    # export PKL file
    create_period_file(cfg, sample, answers, pkl_file)

train size:		121
validation size:	41
test size:		41



### Time series

In [ ]:
# define output folder
path_csv = cfg.path_timeseries / 'csv' #/ cfg.prefix
path_nc = cfg.path_timeseries / 'netcdf' #/ cfg.prefix
for path in [path_csv, path_nc]:
    path.mkdir(parents=True, exist_ok=True)
print(f'Time series will be saved in {cfg.path_timeseries}')

# process timeseries for each station
for ID in tqdm(answers.index, desc='points'):
    
    # TIME SERIES
    # ...........
    try:
        ts = pd.read_parquet(path_in / f'{ID}.parquet')
        ts.dropna(axis=1, how='all', inplace=True)
    except Exception as e:
        logger.error(f'Loading discharge timeseries for station {ID:04d}: {e}')
        continue
    
    # define time period
    start = min(answers.loc[ID, 'start_dates'])
    end = max(answers.loc[ID, 'end_dates'])
    ts = ts.loc[start:end]

    # TEMPORAL ENCODERS
    # .................
    ts['year'] = ts.index.year
    ts['month'] = ts.index.month
    ts['month_sin'], ts['month_cos'] = time_encoding(ts['month'], period=12)
    ts['weekofyear'] = ts.index.isocalendar().week.astype('uint32')
    ts['woy_sin'], ts['woy_cos'] = time_encoding(ts['weekofyear'], period=52)
    ts['dayofyear'] = ts.index.dayofyear
    ts['doy_sin'], ts['doy_cos'] = time_encoding(ts['dayofyear'], period=365)
    ts['dayofweek'] = ts.index.isocalendar().day.astype('uint32')
    ts['dow_sin'], ts['dow_cos'] = time_encoding(ts['dayofweek'], period=7)

    # EXPORT
    # ......
    
    # export CSV file
    ts.to_csv(path_csv / f'{cfg.prefix}_{ID}.csv', index=True)

    # export NetCDF file
    ds = ts.to_xarray()
    ds.attrs['Timezone'] = metadata['Timezone']
    ds.attrs['Sources'] = metadata['Sources']
    for var in ds.data_vars:
        var_short = '_'.join(var.split('_')[:2])  # remove dataset suffix
        if var_short in metadata['variables']:
            ds[var].attrs['long_name'] = metadata['variables'][var_short]['long_name']
            ds[var].attrs['units'] = metadata['variables'][var_short]['units']
    ts.to_xarray().to_netcdf(path_nc / f'{cfg.prefix}_{ID}.nc')

Time series will be saved in /home/casadoj/Data/BEAVERS-ES/v1_0_0/timeseries


points:   0%|          | 0/341 [00:00<?, ?it/s]